In [1]:
import dspy
from typing import Literal, List
dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)


class CLassifyDomain(dspy.Signature):
        """
        Classify the study design described in the abstract.

        Available study-design labels:
            "randomized_controlled_trial", "nonrandomized_controlled_trial",
            "prospective_cohort", "retrospective_cohort", "case_control",
            "cross_sectional", "case_series", "case_report",
            "diagnostic_accuracy_study", "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development", "in_vitro", "animal_model",
            "imaging_only", "conference_abstract_or_poster",
            "other_or_unclear"

        **Primary design selection rules:**
        - Randomized allocation → randomized_controlled_trial
        - Nonrandomized comparator groups → nonrandomized_controlled_trial
        - Prospective follow-up of a group → prospective_cohort
        - Retrospective chart/registry review → retrospective_cohort
        - Explicit “cases vs controls” comparison → case_control
        - Single time-point measurement/survey/prevalence → cross_sectional
        - ≥2 patients without a control group → case_series
        - Single patient → case_report
        - Test/validation of diagnostic performance → diagnostic_accuracy_study
        - Systematic review or meta-analysis → systematic_review_or_meta_analysis
        - No primary data (guidelines, commentary, editorial) → guideline_or_editorial_or_commentary
        - Methods/assay development without clinical outcomes → methods_or_assay_development
        - In vitro experiments → in_vitro
        - Animal experiments → animal_model
        - Imaging-only analyses without clinical outcomes → imaging_only
        - Conference abstracts/posters → conference_abstract_or_poster
        - Anything unclear or mixed → other_or_unclear

        **Secondary design rules:**
        - secondary_designs must be a JSON array.
        - Choose 0–3 additional labels if they meaningfully apply.
        - Use only labels from the primary-design list.
        - Use [] if none apply.

        **Task:**
        Read the abstract and output:
            (1) the single best-fitting primary_design
            (2) an optional list (0–3 items) of secondary_designs
        """
    
        abstract: str = dspy.InputField(
            desc="The Abstract text to classify into themes"
        )
        primary_design: Literal[
            "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
        ] = dspy.OutputField(desc="The primary design classification")
        secondary_designs: List[
            Literal[
                      "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
            ]
        ] = dspy.OutputField(desc="list of 0-3 secondary design classifications")


        

In [2]:
import json
import pandas as pd
with open("../../results/classification/ensembled_classification_results_detailed_filtered.json", "r") as f:
    filtered_data = json.load(f)["results"]

In [4]:
import json
import os

student_llm_string = 'openrouter/google/gemini-2.0-flash-001'
screener_results_path = "../../classifier/domain_classification/classification_results_gemini_groundtruth.json"
results_path = "../../results/domain_classification/classification_results_gemini_original.json"
results_path_save = "../../results/domain_classification/classification_results_gemini_original.jsonl"

API_KEY = os.getenv("openrouter_api_key")
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 350000,
)
dspy.configure(lm=student_lm)

In [7]:
domain_classifier = dspy.ChainOfThought(CLassifyDomain)
domain_classifier.load(path=screener_results_path)
counter = 0
print(len(filtered_data))
for example in filtered_data:
    pred = domain_classifier(abstract=example["abstract"])
    with open(results_path_save, "a") as f:
         f.write(json.dumps({
        "abstract": example["abstract"],
        "primary_design": pred.primary_design,
        "secondary_designs": pred.secondary_designs,
    }) + "\n")
    print(f"Processed {counter} abstracts", end='\r')
    counter += 1
with open(results_path_save, "r") as f:
    data = [json.loads(line) for line in f.readlines()]
with open(results_path, "w") as f:
    json.dump({"results": data}, f, indent=4)

1373


In [19]:
with open("../../results/domain_classification/classification_results_gemini_original.json", "r") as f:
    data_gemini = json.load(f)["results"]
with open("../../results/domain_classification/classification_results_grok_original.json", "r") as f:
    data_grok = json.load(f)["results"]
with open("../../results/domain_classification/classification_results_gpt_original.json", "r") as f:
    data_gpt = json.load(f)["results"]

ensembled_data = []
length = 0
for i in range(len(data_gemini)):
    primary_design_grok = data_grok[i]["primary_design"]
    primary_design_gemini = data_gemini[i]["primary_design"]
    primary_design_gpt = data_gpt[i]["primary_design"]
    if primary_design_grok == primary_design_gemini:
        final_primary_design = primary_design_grok
    elif primary_design_grok == primary_design_gpt:
        final_primary_design = primary_design_grok
    elif primary_design_gemini == primary_design_gpt:
        final_primary_design = primary_design_gemini
    else:
        final_primary_design = "nothing_matched"
    secondary_designs_grok = data_grok[i]["secondary_designs"]
    secondary_designs_gemini = data_gemini[i]["secondary_designs"]
    secondary_designs_gpt = data_gpt[i]["secondary_designs"]
    if secondary_designs_grok is None:
        secondary_designs_grok = []
    if secondary_designs_gemini is None:
        secondary_designs_gemini = []
    if secondary_designs_gpt is None:
        secondary_designs_gpt = []

    final_secondary_designs = list(set(secondary_designs_grok + secondary_designs_gemini + secondary_designs_gpt))

    length += len(final_secondary_designs)
    ensembled_data.append({
        "abstract": data_gemini[i]["abstract"],
        "primary_design": final_primary_design,
        "secondary_designs": final_secondary_designs,
    })
print(f"Average secondary designs length: {length / len(data_gemini):.2f}")
print(f"Maximum secondary designs length: {max([len(item['secondary_designs']) for item in ensembled_data])}")
with open("../../results/domain_classification/ensembled_classification_results_original.json", "w") as f:
    json.dump({"results": ensembled_data}, f, indent=4)

Average secondary designs length: 0.88
Maximum secondary designs length: 5
